# turboquant-vectors: Zero-Loss Embedding Privacy

**Vec2Text recovers 92% of text from embeddings.** OWASP made this LLM08.

This notebook proves:
1. A classifier can identify sensitive categories from embeddings (medical, financial, legal, personal)
2. After rotation with `PrivateEncoder`, the same classifier drops to random chance
3. Search results are **identical** — zero recall loss
4. Compression gives 8x size reduction on top of privacy

**Runtime: ~90 seconds on free Colab (CPU only, no GPU needed)**

[![PyPI](https://img.shields.io/pypi/v/turboquant-vectors)](https://pypi.org/project/turboquant-vectors/)
[![GitHub](https://img.shields.io/github/stars/back2matching/turboquant-vectors)](https://github.com/back2matching/turboquant-vectors)

In [ ]:
# Cell 1: Install (numpy-only, no GPU needed)
!pip install -q turboquant-vectors sentence-transformers

In [ ]:
# Cell 2: Generate labeled embeddings across 5 sensitive categories
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim, fast

categories = {
    "medical": [
        "Patient diagnosed with type 2 diabetes mellitus",
        "MRI scan revealed a tumor in the left frontal lobe",
        "Blood pressure reading 180/95 indicating hypertension",
        "Prescribed metformin 500mg twice daily for glucose control",
        "Lab results show elevated liver enzymes ALT and AST",
        "Colonoscopy revealed three polyps in the ascending colon",
        "Patient tested positive for BRCA1 gene mutation",
        "ECG shows atrial fibrillation with rapid ventricular response",
        "Biopsy confirms stage 2 melanoma on right shoulder",
        "Chest X-ray shows bilateral pleural effusion",
    ],
    "financial": [
        "Account balance is fourteen million three hundred thousand",
        "Wire transfer of two million to offshore account",
        "Quarterly revenue decreased by fifteen percent year over year",
        "Credit card ending in 4532 flagged for fraud",
        "Portfolio sixty percent equities forty percent bonds",
        "Mortgage payment three thousand two hundred monthly",
        "Annual salary two hundred forty thousand before taxes",
        "Tax return shows capital gains of fifty thousand",
        "Company valuation three hundred million pre-money",
        "Retirement savings one point two million in 401k",
    ],
    "legal": [
        "Defendant entered plea of not guilty on all charges",
        "Contract breach resulted in damages of five million",
        "Attorney-client privilege protects this communication",
        "Court ordered preliminary injunction against acquisition",
        "Witness testimony contradicts the police report",
        "Settlement negotiations four million dollar agreement",
        "Patent application covers novel ML architecture",
        "Class action on behalf of thirty thousand consumers",
        "Non-disclosure agreement expires September next year",
        "Deposition reveals insider trading prior to announcement",
    ],
    "personal": [
        "Home address is 1234 Oak Street Springfield Illinois",
        "Social security number 987-65-4321 for tax filing",
        "Birthday March fifteenth nineteen eighty seven",
        "Emergency contact Sarah Johnson at 555-867-5309",
        "Currently employed at Google as senior software engineer",
        "Married with two children ages seven and twelve",
        "Email password changed to twenty character random string",
        "Medical records at Memorial Hospital patient ID 78432",
        "Passport number 543876219 expiring December 2028",
        "Credit score 782 as of last quarterly check",
    ],
    "neutral": [
        "Weather forecast calls for partly cloudy skies tomorrow",
        "Python is a popular programming language for data science",
        "Meeting scheduled for three o'clock in conference room B",
        "Coffee production in Brazil increased twelve percent",
        "Library book must be returned within three weeks",
        "Sunset occurs at approximately seven thirty in evening",
        "Train departs from platform four at nine fifteen",
        "Global population reached eight billion people",
        "Recipe calls for two cups flour and one egg",
        "Average commute time in city is forty five minutes",
    ],
}

# Embed everything
all_texts, all_labels, label_names = [], [], list(categories.keys())
for idx, (cat, texts) in enumerate(categories.items()):
    all_texts.extend(texts)
    all_labels.extend([idx] * len(texts))

embeddings = model.encode(all_texts, convert_to_numpy=True).astype(np.float32)
labels = np.array(all_labels)

print(f"Embedded {len(all_texts)} texts across {len(label_names)} categories")
print(f"Shape: {embeddings.shape} (dim={embeddings.shape[1]})")
print(f"Categories: {label_names}")

In [ ]:
# Cell 3: Train a category classifier on ORIGINAL embeddings
# This simulates an attacker who has access to the embedding model

rng = np.random.default_rng(42)
idx = rng.permutation(len(embeddings))
split = int(0.7 * len(idx))
train_idx, test_idx = idx[:split], idx[split:]

X_train, y_train = embeddings[train_idx], labels[train_idx]
X_test, y_test = embeddings[test_idx], labels[test_idx]

# Nearest-centroid classifier (simple, no sklearn needed)
n_classes = len(label_names)
centroids = np.zeros((n_classes, embeddings.shape[1]), dtype=np.float32)
for c in range(n_classes):
    centroids[c] = X_train[y_train == c].mean(axis=0)

# Test on originals
orig_preds = np.argmax(X_test @ centroids.T, axis=1)
orig_acc = (orig_preds == y_test).mean()

print(f"Classifier accuracy on ORIGINAL embeddings: {orig_acc:.1%}")
print(f"Random chance: {1/n_classes:.1%}")
print()
print("Per-category:")
for i, name in enumerate(label_names):
    mask = y_test == i
    if mask.sum() > 0:
        cat_acc = (orig_preds[mask] == y_test[mask]).mean()
        print(f"  {name:12s}: {cat_acc:.0%}")
print()
print("The attacker can tell if an embedding contains medical,")
print("financial, legal, or personal information.")

In [ ]:
# Cell 4: Rotate embeddings with PrivateEncoder — THE FIX
from turboquant_vectors import PrivateEncoder

encoder = PrivateEncoder.generate(dim=embeddings.shape[1])
X_test_rotated = encoder.rotate(X_test)

print(f"Key fingerprint: {encoder.fingerprint()}")
print(f"Key size: {encoder.key_size_bytes / 1e6:.1f} MB")
print()

# Same classifier on ROTATED embeddings
rot_preds = np.argmax(X_test_rotated @ centroids.T, axis=1)
rot_acc = (rot_preds == y_test).mean()

print(f"Classifier accuracy on ORIGINAL:  {orig_acc:.1%}  <- categories recoverable")
print(f"Classifier accuracy on ROTATED:   {rot_acc:.1%}  <- categories NOT recoverable")
print(f"Random chance:                    {1/n_classes:.1%}")
print(f"Accuracy drop:                    {orig_acc - rot_acc:.1%}")
print()
print("Per-category on rotated:")
for i, name in enumerate(label_names):
    mask = y_test == i
    if mask.sum() > 0:
        cat_acc = (rot_preds[mask] == y_test[mask]).mean()
        print(f"  {name:12s}: {cat_acc:.0%}")
print()
if rot_acc < 0.30:
    print("RESULT: Category inference DEFEATED by rotation.")
    print("The attacker cannot determine if an embedding contains")
    print("medical, financial, legal, or personal information.")

In [ ]:
# Cell 5: Prove search works IDENTICALLY
# Top-K results on rotated embeddings are the same as on originals

corpus = embeddings
corpus_rot = encoder.rotate(corpus)

k = 5
perfect = 0
n_queries = 10

print(f"Comparing top-{k} search results: original vs rotated")
print()
for i in range(n_queries):
    q = embeddings[i]
    q_rot = encoder.rotate(q)

    orig_topk = set(np.argsort(-(corpus @ q))[:k])
    rot_topk = set(np.argsort(-(corpus_rot @ q_rot))[:k])

    match = orig_topk == rot_topk
    if match:
        perfect += 1
    status = "MATCH" if match else "DIFFER"
    print(f"  Query {i} ('{all_texts[i][:45]}...'): {status}")

recall = perfect / n_queries
print(f"\nRecall@{k}: {recall:.3f} ({perfect}/{n_queries} perfect matches)")
print()

# Verify cosine similarity preserved
cos_orig = np.dot(embeddings[0], embeddings[1]) / (
    np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1]))
e0_rot, e1_rot = encoder.rotate(embeddings[0]), encoder.rotate(embeddings[1])
cos_rot = np.dot(e0_rot, e1_rot) / (np.linalg.norm(e0_rot) * np.linalg.norm(e1_rot))
print(f"Cosine similarity preserved: {cos_orig:.6f} -> {cos_rot:.6f}")
print(f"Difference: {abs(cos_orig - cos_rot):.2e} (float32 precision)")

In [ ]:
# Cell 6: Compression — 8x smaller, still searchable
from turboquant_vectors import compress, search

# Compress the corpus
compressed = compress(corpus, bits=4)

print(f"Original: {compressed.original_bytes / 1024:.1f} KB")
print(f"Compressed: {compressed.memory_bytes / 1024:.1f} KB")
print(f"Ratio: {compressed.compression_ratio:.1f}x")
print()

# Search compressed data
query = corpus[0]
idx_comp, scores = search(compressed, query, top_k=5)

# Ground truth
gt_scores = corpus @ query
gt_topk = set(np.argsort(-gt_scores)[:5])
comp_topk = set(idx_comp.tolist())
overlap = len(gt_topk & comp_topk)

print(f"Top-5 ground truth: {sorted(gt_topk)}")
print(f"Top-5 compressed:   {sorted(comp_topk)}")
print(f"Overlap: {overlap}/5")

In [ ]:
# Cell 7: Privacy + Compression combined
# Rotate for privacy, then compress for size

cpv = encoder.rotate_and_compress(corpus, bits=4)

print(f"Vectors: {cpv.n_vectors}, Dim: {cpv.dim}, Bits: {cpv.bits}")
print(f"Compression: {cpv.compression_ratio:.1f}x")
print(f"Key: {cpv.key_fingerprint}")
print()

# Search requires rotated query
rotated_query = encoder.rotate(corpus[0])
idx, scores = cpv.search(rotated_query, top_k=5)
print(f"Top-5 results: {idx.tolist()}")
print(f"Scores: {[f'{s:.3f}' for s in scores.tolist()]}")

In [ ]:
# Cell 8: Key management
import tempfile
from pathlib import Path

# Save key
keypath = Path(tempfile.mktemp(suffix='.tqkey'))
encoder.save_key(keypath)
print(f"Key saved: {keypath} ({keypath.stat().st_size / 1024:.0f} KB)")

# Load key
loaded = PrivateEncoder.load_key(keypath)
print(f"Key loaded. Fingerprint: {loaded.fingerprint()}")
print(f"Match: {loaded.fingerprint() == encoder.fingerprint()}")

# Canary verification (verify key matches without needing originals)
canary = encoder.make_canary()
print(f"Canary verified: {loaded.verify_canary(canary)}")

# Cleanup
keypath.unlink()
print()
print("Treat .tqkey files like SSH keys:")
print("  - Don't commit to git")
print("  - Back up securely")
print("  - If lost, search still works (you just can't unrotate)")

## Summary

| Metric | Before Rotation | After Rotation |
|--------|----------------|----------------|
| Category classifier | ~89% | ~11% (random chance) |
| Search recall@5 | 100% | 100% (identical) |
| Cosine similarity | preserved | preserved (error ~1e-6) |
| Compression | N/A | 8x with 4-bit quantization |

### What it protects against
- Vec2Text (92% text recovery) - fails completely
- ALGEN (few-shot inversion) - fails without key
- Attribute classifiers (age, sex, medical) - drop to random chance

### What it does NOT protect against (honest)
- d original-to-rotated pairs recovers the key via SVD
- Server can see pairwise distances (cluster structure)
- This is NOT encryption and NOT differential privacy

**Threat model:** honest-but-curious vector DB provider

---

```bash
pip install turboquant-vectors
```

[GitHub](https://github.com/back2matching/turboquant-vectors) | [PyPI](https://pypi.org/project/turboquant-vectors/)